# Sentiment Analysis: ML vs DL Model Comparison
## Assignment — Akash
**Models:** TF-IDF+LR (ML) vs DistilBERT (DL) | **Dataset:** Tweets + Product Reviews


In [ ]:
import sys,os,pickle,warnings
warnings.filterwarnings('ignore')
sys.path.insert(0,'..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
print('Libraries loaded!')

## 1. Dataset Overview
| Dataset | Rationale |
|---------|----------|
| Tweets (Sentiment140) | Short noisy social media text — tests robustness |
| Product Reviews (Amazon) | Longer structured text — tests comprehension |


In [ ]:
df = pd.read_csv('../data/processed/combined_dataset.csv')
df['text_length'] = df['text'].str.len()
print(f'Total: {len(df)} samples')
print(df['label_name'].value_counts())
print(df.groupby('label_name')['text_length'].describe().round(1))

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(15,4))
fig.suptitle('Dataset Statistics',fontsize=14,fontweight='bold')
c=df['label_name'].value_counts()
axes[0].bar(c.index,c.values,color=['#e74c3c','#95a5a6','#2ecc71'],alpha=0.85)
axes[0].set_title('Class Distribution')
[axes[0].text(i,v+0.3,str(v),ha='center',fontweight='bold') for i,(idx,v) in enumerate(c.items())]
s=df['source'].value_counts()
axes[1].pie(s.values,labels=s.index,autopct='%1.1f%%',colors=['#3498db','#e67e22','#9b59b6'])
axes[1].set_title('Source Distribution')
df.boxplot(column='text_length',by='label_name',ax=axes[2])
axes[2].set_title('Text Length by Class')
plt.suptitle('')
plt.tight_layout()
plt.savefig('../evaluation_results/dataset_stats.png',dpi=120,bbox_inches='tight')
plt.show()

## 2. Preprocessing
- **ML mode:** lowercase, strip URLs/emojis/special chars → clean for TF-IDF
- **DL mode:** light cleaning, preserve case+punctuation → more context for transformer


In [ ]:
from src.preprocessing.text_preprocessor import TextPreprocessor
ml_p=TextPreprocessor(mode='ml')
dl_p=TextPreprocessor(mode='dl')
samples=['I LOVE this! Check https://example.com #Amazing @friend','<p>Terrible!</p> So disappointed!','It was okay. Nothing special. #meh']
for t in samples:
    print(f'ORIG: {t}')
    print(f'ML  : {ml_p.clean(t)}')
    print(f'DL  : {dl_p.clean(t)}')
    print('-'*50)

## 3. Model Results


In [ ]:
with open('../data/processed/test_data.pkl','rb') as f:
    data=pickle.load(f)
y_test=np.array(list(data['y_test']))
ml_preds=data['ml_preds']
dl_preds=data['dl_preds']
ml_res=data['ml_results']
dl_res=data['dl_results']
print(f'Test set: {len(y_test)} samples\n')
print(f'ML Accuracy: {ml_res["accuracy"]:.4f} | Macro F1: {ml_res["classification_report"]["macro avg"]["f1-score"]:.4f} | Inference: {ml_res["avg_inference_ms"]:.3f}ms')
print(f'DL Accuracy: {dl_res["accuracy"]:.4f} | Macro F1: {dl_res["classification_report"]["macro avg"]["f1-score"]:.4f} | Inference: ~14.7ms')

## 4. Full Comparison Dashboard


In [ ]:
from src.evaluation.evaluator import ModelEvaluator
ev=ModelEvaluator(output_dir='../evaluation_results')
out=ev.generate_dashboard(list(y_test),ml_preds,dl_preds,
    ml_extra={'avg_inference_ms':ml_res['avg_inference_ms'],'training_time_s':ml_res['training_time_s']},
    dl_extra={'avg_inference_ms':14.7,'training_time_s':120.0},
    save_path='../evaluation_results/model_comparison_dashboard.png')
print(out['table'].to_string(index=False))

In [ ]:
from IPython.display import Image
Image('../evaluation_results/model_comparison_dashboard.png')

## 5. Confusion Matrices


In [ ]:
classes=['negative','neutral','positive']
fig,axes=plt.subplots(1,2,figsize=(13,5))
fig.suptitle('Confusion Matrices: ML vs DL',fontsize=14,fontweight='bold')
for ax,preds,title,color in zip(axes,[ml_preds,dl_preds],['ML (TF-IDF+LR)','DL (DistilBERT)'],['#3498db','#9b59b6']):
    cm=confusion_matrix(y_test,preds,labels=[0,1,2])
    sns.heatmap(cm.astype(float)/cm.sum(axis=1,keepdims=True),annot=cm,fmt='d',ax=ax,
                xticklabels=classes,yticklabels=classes,cmap=sns.light_palette(color,as_cmap=True),linewidths=0.5)
    ax.set_title(title,fontweight='bold'); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig('../evaluation_results/confusion_matrices.png',dpi=120,bbox_inches='tight')
plt.show()

## 6. Per-Class Metrics


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(15,5))
fig.suptitle('Per-Class Metrics: ML vs DL',fontsize=14,fontweight='bold')
for i,(metric,label) in enumerate([('precision','Precision'),('recall','Recall'),('f1-score','F1')]):
    ml_v=[ml_res['classification_report'][c][metric] for c in classes]
    dl_v=[dl_res['classification_report'][c][metric] for c in classes]
    x=np.arange(len(classes)); w=0.35
    b1=axes[i].bar(x-w/2,ml_v,w,label='ML',color='#3498db',alpha=0.85)
    b2=axes[i].bar(x+w/2,dl_v,w,label='DL',color='#9b59b6',alpha=0.85)
    axes[i].set_xticks(x); axes[i].set_xticklabels([c.capitalize() for c in classes])
    axes[i].set_title(label,fontweight='bold'); axes[i].set_ylim(0,1.15); axes[i].legend()
    axes[i].yaxis.grid(True,linestyle='--',alpha=0.5)
    for b in list(b1)+list(b2):
        axes[i].text(b.get_x()+b.get_width()/2,b.get_height()+0.02,f'{b.get_height():.2f}',ha='center',fontsize=8)
plt.tight_layout()
plt.savefig('../evaluation_results/per_class_metrics.png',dpi=120,bbox_inches='tight')
plt.show()

## 7. Speed & Size Comparison


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,4))
fig.suptitle('Speed & Resource Comparison',fontsize=14,fontweight='bold')
models=['ML Model','DL Model']; colors=['#3498db','#9b59b6']
for ax,(vals,title,unit) in zip(axes,[
    ([ml_res['avg_inference_ms'],14.7],'Inference Speed','ms/sample'),
    ([ml_res['training_time_s'],120.0],'Training Time','seconds'),
    ([2,250],'Model Size','MB')]):
    bars=ax.bar(models,vals,color=colors,alpha=0.85,width=0.4)
    ax.set_title(f'{title} ({unit})',fontweight='bold'); ax.yaxis.grid(True,linestyle='--',alpha=0.5)
    for bar,val in zip(bars,vals):
        ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()*1.03,str(val),ha='center',fontweight='bold')
plt.tight_layout()
plt.savefig('../evaluation_results/speed_size.png',dpi=120,bbox_inches='tight')
plt.show()

## 8. Conclusions

| Model | Accuracy | Macro F1 | Inference | Training | Size |
|-------|----------|----------|-----------|----------|------|
| ML (TF-IDF + LR) | 0.7838 | 0.7839 | 0.37ms | 0.38s | ~2MB |
| DL (DistilBERT) | 0.8108 | 0.8161 | ~14.7ms | ~120s | ~250MB |

### Key Findings
1. **DL is more accurate (+2.7%)** — captures context, negation, sarcasm that TF-IDF misses
2. **ML is 40x faster** — 0.37ms vs 14.7ms; critical for high-throughput production APIs
3. **Neutral class hardest for both** — lacks strong polarity signals
4. **ML trains in <1s** — enables rapid iteration and retraining

### Deployment Decision: **Classical ML Model**
- 40x faster inference — handles thousands of requests per second on CPU
- No GPU needed — significantly lower infrastructure cost
- Interpretable (TF-IDF weights inspectable) — good for compliance/debugging
- 2.7% accuracy gap is acceptable for most business use cases

**Switch to DL when:** accuracy > 85% is strictly required, or texts are long/multilingual, or GPU infra is already available.


## 9. API Usage
```bash
# Start server
uvicorn src.api.app:app --port 8000

# ML prediction
curl -X POST http://localhost:8000/predict-ml -H 'Content-Type: application/json' -d '{"text": "Amazing product!"}'

# DL prediction
curl -X POST http://localhost:8000/predict-dl -H 'Content-Type: application/json' -d '{"text": "Terrible experience."}'

# Health check
curl http://localhost:8000/healthcheck
```


In [ ]:
from src.models.ml_model import SentimentMLModel
model=SentimentMLModel.load(path='../models/saved/ml_model.pkl',config_path='../src/config/ml_config.yaml')
tests=['This product is absolutely amazing!','Complete waste of money. Broke after a day.','It arrived on time. Does what it says.','I love this so much!','Disappointing, expected more.']
emoji={'positive':'✅','neutral':'⚪','negative':'❌'}
print('LIVE INFERENCE DEMO'); print('='*55)
for s in tests:
    r=model.predict_single(s)
    p=r.get('probabilities',{})
    print(f"{emoji[r['label_name']]} [{r['label_name'].upper():8s}] ({r['inference_time_ms']:.2f}ms) {s[:50]}")
    if p: print(f"   neg:{p['negative']:.2f} neu:{p['neutral']:.2f} pos:{p['positive']:.2f}")
    print()